# 1 Explore and preprocess data
Iterate over raw nights, clean up sensors seperately, resaple & synchronize, then merge.

## Imports

In [34]:
import pandas as pd
from pathlib import Path
import os
import datetime
pd.set_option('display.max_columns', None)

## Iterate over nights and show data 


In [35]:
# Load path to raw data
data_dir = Path('../data/raw')

# Get all night folders
night_folders = sorted([
    d for d in os.listdir(data_dir)
    if (data_dir / d).is_dir()
])

print(f"{len(night_folders)} nights found:")
print(night_folders)

# Select one night
night_id = night_folders[1]   # change index if needed

print(f"\nProcessing night: {night_id}")

# Paths
accel_path = data_dir / night_id / 'Accelerometer.csv'
gyro_path = data_dir / night_id / 'Gyroscope.csv'

# Check if files exist
if not accel_path.exists() or not gyro_path.exists():
    print("Missing files.")
else:
    # Load data
    df_accel = pd.read_csv(accel_path)
    df_gyro = pd.read_csv(gyro_path)

    # Shapes
    print(f"\nAccelerometer shape: {df_accel.shape}")
    print(f"Gyroscope shape: {df_gyro.shape}")

    # Preview
    print("\nAccelerometer head:")
    display(df_accel.head())

    print("\nGyroscope head:")
    display(df_gyro.head())

6 nights found:
['2026-04-21_22-34-53', '2026-04-28_22-22-45', '2026-04-30_23-05-56', '2026-05-01_23-22-33', '2026-05-02_21-05-13', '2026-05-03_22-37-52']

Processing night: 2026-04-28_22-22-45

Accelerometer shape: (2704528, 5)
Gyroscope shape: (2722658, 5)

Accelerometer head:


,time,seconds_elapsed,z,y,x
0,1777414965293580300,0.17358,-0.045000,0.258697,-0.045120
1,1777414965313580300,0.19358,-0.064801,0.233745,0.019503
2,1777414965323580400,0.20358,0.010445,0.190032,-0.000851
3,1777414965333580300,0.21358,-0.031610,0.158273,-0.004397
4,1777414965343580200,0.22358,-0.062271,0.073154,0.037912



Gyroscope head:


,time,seconds_elapsed,z,y,x
0,1777414965283580400,0.163580,-0.014813,0.079642,-0.073609
1,1777414965316611000,0.196611,-0.014813,0.079642,-0.073609
2,1777414965326611000,0.206611,-0.014813,0.079642,-0.073609
3,1777414965336611000,0.216611,0.037110,0.105909,0.024129
4,1777414965346610700,0.226611,0.041997,0.093081,0.036957


*=> Sensors have a different amount of rows, merging by row index therefore not possible.*  
*Next step: Find out if timestamp of start and stop align, check average timestamp and total duration.*

### Diagnostics

In [37]:
# Start time
print(f"\nStarting time Accelerometer: {pd.to_datetime(df_accel['time'].iloc[0])}")
print(f"Starting time Gyroscope: {pd.to_datetime(df_gyro['time'].iloc[0])}")

# End time
print(f"\nEnd time Accelerometer: {pd.to_datetime(df_accel['time'].iloc[-1])}")
print(f"End time Gyroscope: {pd.to_datetime(df_gyro['time'].iloc[-1])}")

# Start-time difference
start_diff = (
    pd.to_datetime(df_accel['time'].iloc[0]) -
    pd.to_datetime(df_gyro['time'].iloc[0])
)

print(f"\nStart-time difference: {abs(start_diff)}")

# End-time difference
end_diff = (
    pd.to_datetime(df_accel['time'].iloc[-1]) -
    pd.to_datetime(df_gyro['time'].iloc[-1])
)

print(f"End-time difference: {abs(end_diff)}")

# Recording duration
accel_duration = (
    pd.to_datetime(df_accel['time'].iloc[-1]) -
    pd.to_datetime(df_accel['time'].iloc[0])
).total_seconds()

gyro_duration = (
    pd.to_datetime(df_gyro['time'].iloc[-1]) -
    pd.to_datetime(df_gyro['time'].iloc[0])
).total_seconds()

print(f"\nAccelerometer duration (s): {accel_duration:.2f}")
print(f"Gyroscope duration (s): {gyro_duration:.2f}")

# Effective sampling frequency
accel_freq = len(df_accel) / accel_duration
gyro_freq = len(df_gyro) / gyro_duration

print(f"\nEffective Accelerometer frequency: {accel_freq:.2f} Hz")
print(f"Effective Gyroscope frequency: {gyro_freq:.2f} Hz")

# Mean timestep
accel_dt = df_accel['seconds_elapsed'].diff().mean()
gyro_dt = df_gyro['seconds_elapsed'].diff().mean()

print(f"\nMean Accelerometer timestep: {accel_dt:.6f} s")
print(f"Mean Gyroscope timestep: {gyro_dt:.6f} s")



Starting time Accelerometer: 2026-04-28 22:22:45.293580300
Starting time Gyroscope: 2026-04-28 22:22:45.283580400

End time Accelerometer: 2026-04-29 06:00:25.046073
End time Gyroscope: 2026-04-29 06:00:25.046073

Start-time difference: 0 days 00:00:00.009999900
End-time difference: 0 days 00:00:00

Accelerometer duration (s): 27459.75
Gyroscope duration (s): 27459.76

Effective Accelerometer frequency: 98.49 Hz
Effective Gyroscope frequency: 99.15 Hz

Mean Accelerometer timestep: 0.010153 s
Mean Gyroscope timestep: 0.010086 s


*=> Negligible sampling frequency, overall stable and synchronized*  
*Next steps: 
1. Decide on resample 100Hz OR timestamp aware nearest merge. 
2. Timestamp consistency
3. Labels?
4. Sensor distributions
5. Visualize small segments